# E-Commerce Sales Analysis — Day 2: Data Exploration

## Objective
Initial profiling of all 9 Olist datasets to understand structure, scale, 
and data quality before deep analysis.

## Datasets
1. **orders** - 99,441 orders with timestamps and status
2. **order_items** - 112,650 line items with prices
3. **products** - 32,951 products with category info
4. **customers** - 99,441 customer records
5. **sellers** - 3,095 sellers
6. **payments** - 103,886 payment records
7. **reviews** - 99,224 reviews with scores and comments
8. **geolocation** - ~1M zip code coordinates
9. **category_translation** - 71 categories (PT to EN translation)

## Time Span
Sept 2016 to Oct 2018 (~2 years of data)

## Initial Observations

### Data Scale and Coverage
- 99,441 orders over 772 days (Sept 2016 to Oct 2018), which works out to roughly 129 orders per day on average
- 96,096 unique customers across 99,441 customer records, so only about 3.5% of customers ordered more than once. That's a low repeat rate worth digging into
- order_items has 112,650 rows vs 99,441 orders, meaning about 13% of orders have multiple items
- payments has 103,886 rows vs 99,441 orders, so around 4.5% of orders use split payments (multiple payment methods or installments)
- geolocation has 1,000,163 rows but only ~99K customers exist. This is clearly a many-to-one zip code lookup table, not a customer record table

### Data Quality and Missing Values
- order_delivered_customer_date has 2,965 missing (3.0%). These are likely cancelled orders or orders still in transit at the time of extraction
- product_category_name has 610 missing (1.9%), so 610 products will need to be excluded or imputed for any category-level analysis
- Review comment titles are 88.3% missing and review messages are 58.7% missing. This isn't a data quality issue, it's normal user behavior. Most people leave a star rating without writing anything
- order_items, customers, sellers, payments, and geolocation are all clean with zero missing values
- All date columns are stored as strings (object dtype). Need to convert these before any time series analysis

### Structural Observations
- order_id is the main join key linking orders to order_items, payments, and reviews
- customer_id (one per order) is different from customer_unique_id (one per person). Important distinction for any retention or lifetime value work
- All 73 product categories are in Portuguese, so the category_translation table will be essential for English labels in the dashboard
- The category_translation table only has 71 entries, but products contains 73 unique categories. Two categories don't have English translations and will need to be handled
- Both customers and sellers join to geolocation through zip_code_prefix
- Reviews link via order_id, not directly to customer_id. To connect review sentiment to repeat behavior, I'll need to join through orders first

### Business Context
- Total revenue: 13,591,643.70 BRL, which is roughly ₹20.4 crores or $2.7M USD over two years
- Total freight: 2,251,909.54 BRL, working out to about 16.6% of revenue. That feels high for e-commerce, where typical freight costs run 8-15%. Worth investigating by category
- Average order value is around 138 BRL, or about ₹2,055. This positions Olist as a mid-tier marketplace, not luxury and not budget
- 3,095 sellers serving 96K unique customers, so each seller serves about 31 customers on average
- 32,951 products spread across 73 categories, averaging around 451 products per category

### Open Questions for Deeper Analysis
- Why is the repeat customer rate only 3.5%? Is this a structural issue with the marketplace, a measurement artifact (data extracted before customers had time to come back), or genuinely poor retention?
- Are reviews skewed toward extremes? People tend to leave reviews when they're either very happy or very angry. Worth checking the distribution.
- What does seller concentration look like? Doing a Pareto check to see if the top 20% of sellers drive 80% of GMV
- Which two categories are missing translations, and how big are they? Could affect what shows up in the final dashboard
- Is the 2017 to 2018 growth real, or is it inflated because 2016 only has partial data?
- Freight at 16.6% of revenue is high. Is that consistent across categories, or are heavy items dragging the average up?

## Next Steps
- Build entity relationship diagram between tables
- Begin analysis of Question 1 (revenue and growth by category)
- Set up the AI integration for semantic clustering

In [1]:
# E-commerce Sales Analysis — Data Exploration
# Author: Rohan Vishwakarma
# Day 2: Initial data loading and profiling

import pandas as pd
import numpy as np
from pathlib import Path

# Display settings — show all columns, decent width
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 100)

# Path to our raw data
DATA_DIR = Path('../data/raw')

print("Setup complete ✅")
print(f"Data directory: {DATA_DIR.absolute()}")

Setup complete ✅
Data directory: /Users/rohanvishwakarma/Developer/analytics-portfolio/01-ecommerce-sales-analysis/notebooks/../data/raw


In [2]:
# Load all 9 Olist CSVs

orders = pd.read_csv(DATA_DIR / 'olist_orders_dataset.csv')
order_items = pd.read_csv(DATA_DIR / 'olist_order_items_dataset.csv')
products = pd.read_csv(DATA_DIR / 'olist_products_dataset.csv')
customers = pd.read_csv(DATA_DIR / 'olist_customers_dataset.csv')
sellers = pd.read_csv(DATA_DIR / 'olist_sellers_dataset.csv')
payments = pd.read_csv(DATA_DIR / 'olist_order_payments_dataset.csv')
reviews = pd.read_csv(DATA_DIR / 'olist_order_reviews_dataset.csv')
geolocation = pd.read_csv(DATA_DIR / 'olist_geolocation_dataset.csv')
category_translation = pd.read_csv(DATA_DIR / 'product_category_name_translation.csv')

print("All 9 datasets loaded successfully ✅")

All 9 datasets loaded successfully ✅


In [3]:
# Quick sanity check — show the first 5 rows of orders

orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [4]:
# How big is each table?

datasets = {
    'orders': orders,
    'order_items': order_items,
    'products': products,
    'customers': customers,
    'sellers': sellers,
    'payments': payments,
    'reviews': reviews,
    'geolocation': geolocation,
    'category_translation': category_translation,
}

print(f"{'Table':<25} {'Rows':>10} {'Columns':>10}")
print("-" * 50)
for name, df in datasets.items():
    print(f"{name:<25} {len(df):>10,} {len(df.columns):>10}")

Table                           Rows    Columns
--------------------------------------------------
orders                        99,441          8
order_items                  112,650          7
products                      32,951          9
customers                     99,441          5
sellers                        3,095          4
payments                     103,886          5
reviews                       99,224          7
geolocation                1,000,163          5
category_translation              71          2


In [5]:
# Inspect each table's columns and data types

for name, df in datasets.items():
    print(f"\n{'='*60}")
    print(f"📊 {name.upper()}")
    print(f"{'='*60}")
    print(f"Shape: {df.shape}")
    print(f"\nColumns and types:")
    print(df.dtypes)
    print(f"\nFirst 3 rows:")
    print(df.head(3))


📊 ORDERS
Shape: (99441, 8)

Columns and types:
order_id                         str
customer_id                      str
order_status                     str
order_purchase_timestamp         str
order_approved_at                str
order_delivered_carrier_date     str
order_delivered_customer_date    str
order_estimated_delivery_date    str
dtype: object

First 3 rows:
                           order_id                       customer_id order_status order_purchase_timestamp    order_approved_at order_delivered_carrier_date order_delivered_customer_date  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d    delivered      2017-10-02 10:56:33  2017-10-02 11:07:15          2017-10-04 19:55:00           2017-10-10 21:25:13   
1  53cdb2fc8bc7dce0b6741e2150273451  b0830fb4747a6c6d20dea0b8c802d7ef    delivered      2018-07-24 20:41:37  2018-07-26 03:24:27          2018-07-26 14:31:00           2018-08-07 15:27:45   
2  47770eb9100c2d0c44946d9cf07ec65d  41ce2a54c0b03bf34

In [6]:
# How much data is missing in each table?

print("Missing values per table:\n")
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]  # only show columns with missing values
    if len(missing) > 0:
        print(f"\n{name}:")
        for col, count in missing.items():
            pct = (count / len(df)) * 100
            print(f"  {col}: {count:,} missing ({pct:.1f}%)")
    else:
        print(f"{name}: ✅ no missing values")

Missing values per table:


orders:
  order_approved_at: 160 missing (0.2%)
  order_delivered_carrier_date: 1,783 missing (1.8%)
  order_delivered_customer_date: 2,965 missing (3.0%)
order_items: ✅ no missing values

products:
  product_category_name: 610 missing (1.9%)
  product_name_lenght: 610 missing (1.9%)
  product_description_lenght: 610 missing (1.9%)
  product_photos_qty: 610 missing (1.9%)
  product_weight_g: 2 missing (0.0%)
  product_length_cm: 2 missing (0.0%)
  product_height_cm: 2 missing (0.0%)
  product_width_cm: 2 missing (0.0%)
customers: ✅ no missing values
sellers: ✅ no missing values
payments: ✅ no missing values

reviews:
  review_comment_title: 87,656 missing (88.3%)
  review_comment_message: 58,247 missing (58.7%)
geolocation: ✅ no missing values
category_translation: ✅ no missing values


In [7]:
# Convert date columns and check the time span

# Convert order timestamps to datetime
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors='coerce')

# Get the date range
print(f"Earliest order: {orders['order_purchase_timestamp'].min()}")
print(f"Latest order:   {orders['order_purchase_timestamp'].max()}")
print(f"Time span:      {orders['order_purchase_timestamp'].max() - orders['order_purchase_timestamp'].min()}")

Earliest order: 2016-09-04 21:15:19
Latest order:   2018-10-17 17:30:18
Time span:      772 days 20:14:59


In [8]:
# Quick top-level metrics

total_orders = len(orders)
unique_customers = customers['customer_unique_id'].nunique()
total_revenue = order_items['price'].sum()
total_freight = order_items['freight_value'].sum()
unique_sellers = sellers['seller_id'].nunique()
unique_products = products['product_id'].nunique()
unique_categories = products['product_category_name'].nunique()

print("📈 TOP-LEVEL METRICS")
print("=" * 50)
print(f"Total orders:        {total_orders:>15,}")
print(f"Unique customers:    {unique_customers:>15,}")
print(f"Total revenue (BRL): {total_revenue:>15,.2f}")
print(f"Total freight (BRL): {total_freight:>15,.2f}")
print(f"Unique sellers:      {unique_sellers:>15,}")
print(f"Unique products:     {unique_products:>15,}")
print(f"Product categories:  {unique_categories:>15}")

📈 TOP-LEVEL METRICS
Total orders:                 99,441
Unique customers:             96,096
Total revenue (BRL):   13,591,643.70
Total freight (BRL):    2,251,909.54
Unique sellers:                3,095
Unique products:              32,951
Product categories:               73
